In [26]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "precio_noche",
        "puntuacion",
        "relacion_calidad_precio"
    ],
    outputCol="features"
)

df_features = assembler.transform(df_cluster)

In [27]:
from pyspark.ml.clustering import KMeans

kmeans = KMeans(
    k=3,
    seed=1,
    featuresCol="features"
)

modelo = kmeans.fit(df_features)

In [28]:
df_resultado = modelo.transform(df_features)

df_resultado.select(
    "nombre_hotel",
    "precio_noche",
    "puntuacion",
    "relacion_calidad_precio",
    "prediction"
).show(20, truncate=False)

+------------------------------------------------------------+------------+----------+-----------------------+----------+
|nombre_hotel                                                |precio_noche|puntuacion|relacion_calidad_precio|prediction|
+------------------------------------------------------------+------------+----------+-----------------------+----------+
|Hotel Diego de Velazquez                                    |159716.0    |8.4       |0.5259335320193343     |0         |
|Hotel Diego de Almagro Viña del Mar                         |310485.0    |8.7       |0.2802067732740712     |2         |
|Alto Chinchorro Hostel                                      |53239.0     |8.4       |1.5777907173312797     |0         |
|Hotel Arica                                                 |168082.0    |8.2       |0.4878571173593841     |2         |
|MR Hotel Providencia (ex Hotel Neruda)                      |196223.0    |8.6       |0.43827685847224834    |2         |
|Hostal Casa Azul       

In [29]:
df_resultado.groupBy("prediction").count().show()

+----------+-----+
|prediction|count|
+----------+-----+
|         1|  110|
|         2|  775|
|         0| 2261|
+----------+-----+



In [30]:
centros = modelo.clusterCenters()

for i, centro in enumerate(centros):
    print(f"Cluster {i}: {centro}")

Cluster 0: [8.48744710e+04 7.23366210e+00 1.01238482e+00]
Cluster 1: [6.01002673e+05 5.91684545e+00 1.08759692e-01]
Cluster 2: [2.41370059e+05 7.48818065e+00 3.30703133e-01]


In [31]:
from pyspark.sql.functions import avg

df_resultado.groupBy("prediction").agg(
    avg("precio_noche").alias("precio_promedio"),
    avg("puntuacion").alias("nota_promedio"),
    avg("relacion_calidad_precio").alias("conveniencia_promedio")
).show(truncate=False)

+----------+-----------------+-----------------+---------------------+
|prediction|precio_promedio  |nota_promedio    |conveniencia_promedio|
+----------+-----------------+-----------------+---------------------+
|1         |601002.6727272727|5.916845454545455|0.10875969209004824  |
|2         |241370.0593548387|7.48818064516129 |0.3307031326106945   |
|0         |84874.47103051747|7.233662096417518|1.0123848237000146   |
+----------+-----------------+-----------------+---------------------+



In [32]:
df_guardar2.printSchema()

root
 |-- _id: string (nullable = true)
 |-- adultos: integer (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- estrellas: integer (nullable = true)
 |-- estudiante: string (nullable = true)
 |-- evaluacion_conveniencia: string (nullable = true)
 |-- fecha_captura: timestamp (nullable = true)
 |-- grupo: string (nullable = true)
 |-- integrante: string (nullable = true)
 |-- noches: integer (nullable = true)
 |-- nombre_hotel: string (nullable = true)
 |-- plataforma: string (nullable = true)
 |-- precio_noche: double (nullable = true)
 |-- precio_usd: double (nullable = true)
 |-- puntuacion: double (nullable = true)
 |-- relacion_calidad_precio: double (nullable = true)
 |-- segmento_precio: string (nullable = true)
 |-- tipo_alojamiento: string (nullable = true)
 |-- tipo_habitacion: string (nullable = true)
 |-- url_origen: string (nullable = true)
 |-- zona_geografica: string (nullable = true)
 |-- prediction: integer (nullable = false)



In [33]:
df_guardar2.groupBy("prediction").count().show()

+----------+-----+
|prediction|count|
+----------+-----+
|         1|  110|
|         2|  775|
|         0| 2261|
+----------+-----+



In [34]:
df_guardar2.groupBy("prediction").agg(
    avg("precio_noche"),
    avg("puntuacion"),
    avg("relacion_calidad_precio")
).show()

+----------+-----------------+-----------------+----------------------------+
|prediction|avg(precio_noche)|  avg(puntuacion)|avg(relacion_calidad_precio)|
+----------+-----------------+-----------------+----------------------------+
|         1|601002.6727272727|5.916845454545455|         0.10875969209004824|
|         2|241370.0593548387| 7.48818064516129|          0.3307031326106945|
|         0|84874.47103051747|7.233662096417518|          1.0123848237000146|
+----------+-----------------+-----------------+----------------------------+



In [ ]:
Aprendizaje No Supervisado (Clustering) HITO 2

Con el objetivo de identificar patrones ocultos en los datos, se aplicó el algoritmo K-Means, una técnica de aprendizaje no supervisado 
que permite agrupar alojamientos con características similares sin necesidad de etiquetas previas.

Para el análisis se utilizaron las variables precio por noche, puntuación y relación calidad-precio, ya que representan los factores más relevantes 
para los usuarios al momento de elegir un alojamiento.

El modelo generó distintos clusters, asignando cada alojamiento a un grupo según sus características. Gracias a esta segmentación 
fue posible diferenciar alojamientos económicos con buena relación calidad-precio, alojamientos premium con precios más elevados y opciones intermedias.

Los resultados obtenidos permiten comprender mejor el comportamiento del mercado turístico chileno y facilitan la identificación de alojamientos que
ofrecen mayor valor para los usuarios.

Conclusión del Clustering

La aplicación de K-Means permitió segmentar exitosamente los alojamientos de la base de datos en grupos con características similares. 
Esta clasificación aporta información valiosa para apoyar la toma de decisiones de los usuarios, ya que permite identificar de manera rápida qué
alojamientos destacan por su conveniencia, calidad o exclusividad, cumpliendo así con el objetivo planteado en el proyecto.